# ipscout Quickstart

Install the package (from GitHub or PyPI), then try the CLI and library examples below. Run the
cells in order from the top: later cells use `sys` and `ipscout` imported above them. Every
probe targets this machine's own loopback addresses, so nothing leaves the host.

In [ ]:
# The install cells below need this kernel's own interpreter, whichever one you use.
import sys

## 1) Install latest from GitHub (default)

In [ ]:
# Install uv into this kernel's environment first: a bare `!uv` needs uv on
# PATH, which a reader running this notebook from a plain Python will not have.
!{sys.executable} -m pip install --quiet --upgrade uv
assert _exit_code == 0

# install ipscout from GitHub (default)
!{sys.executable} -m uv pip install --python {sys.executable} --upgrade git+https://github.com/bitranox/ipscout.git
assert _exit_code == 0

## Or: Install from PyPI (alternative)
Set `USE_PYPI = True` and run the cell below if you prefer the PyPI release.

In [ ]:
USE_PYPI = False

if USE_PYPI:
    !{sys.executable} -m pip install --upgrade ipscout
    assert _exit_code == 0

## 2) CLI: version, help and capabilities
`python -m ipscout` is the same CLI as the `ipscout` command, and works even when the kernel's
`bin` directory is not on `PATH`. `capabilities` reports what this host can do unprivileged.

In [ ]:
!{sys.executable} -m ipscout --version
assert _exit_code == 0
!{sys.executable} -m ipscout --help
assert _exit_code == 0
!{sys.executable} -m ipscout info
assert _exit_code == 0
!{sys.executable} -m ipscout capabilities
assert _exit_code == 0

## 3) Library: ping loopback
`raise_on_error=False` reports a failure on the result instead of raising, so on a host where
unprivileged ICMP is switched off (see `docs/installation.md`) this shows the reason.

In [ ]:
import ipscout

result = ipscout.ping("127.0.0.1", 2, interval=0, raise_on_error=False)
result.reached, result.method.value, result.str_result, result.error

`icmp_available()` asks the host whether unprivileged ICMP works. `is_reachable` never raises
and falls back to TCP.

In [ ]:
ipscout.icmp_available(), ipscout.is_reachable("127.0.0.1")

## 4) Library: sweep several hosts
A notebook already runs an event loop, so a sweep here awaits `aping_many`; in a plain script
the synchronous `ping_many` takes the same arguments. Either one reports a failure on each
result rather than raising, so one unreachable target does not stop the sweep.

In [ ]:
results = await ipscout.aping_many(["127.0.0.1", "::1"], times=1)
{target: (r.reached, r.error) for target, r in results.items()}

## 5) Library: local interfaces

In [ ]:
[(item.name, item.index, item.is_up, item.is_loopback) for item in ipscout.local_interfaces()]

## 6) Library: results are frozen Pydantic models
`model_dump()` carries every derived statistic, so a result serialises without losing
anything.

In [ ]:
result.model_dump(mode="json")